# DAS pick-quality screening

Use this notebook after ELEP has exported pick CSVs. It calls the existing `das_event_catalog.py` without duplicating its criteria. It does not filter waveforms or rerun ELEP, locate verified earthquakes, or compare with ComCat. Read `ML/README.md` for thresholds and limitations.


## 1. Configure paths

Run from the repository root or ML directory. Set an existing input folder and a NEW output folder so earlier results are not overwritten. Only matching MORADAS S-pick CSVs will be processed.


In [ ]:
from pathlib import Path
import subprocess
import sys
import csv

ROOT = Path.cwd()
if not (ROOT / "ML").is_dir():
    ROOT = ROOT.parent
SCRIPT = ROOT / "ML" / "das_event_catalog.py"
INPUT_DIR = Path("/path/to/pick_csvs")
OUTPUT_DIR = Path("/path/to/new_results")
PATTERN = "MORADAS_*_AR.csv"


## 2. Run screening

The script finds spatially continuous picks, tests local smoothness, groups blocks in time and fits relative S-wave moveout. A common time offset avoids relying on the absolute DAS clock. These empirical tests reject incoherent candidates; passing them does not independently confirm an earthquake. The physical search can take several minutes per dataset.


In [ ]:
assert SCRIPT.is_file(), f"Script not found: {SCRIPT}"
assert INPUT_DIR.is_dir(), "Set INPUT_DIR to the existing pick folder."
assert not OUTPUT_DIR.exists(), "Choose a new output folder to preserve previous results."
assert any(INPUT_DIR.rglob(PATTERN)), f"No files match {PATTERN}"
subprocess.run([sys.executable, str(SCRIPT), "--input-dir", str(INPUT_DIR),
                "--output-dir", str(OUTPUT_DIR), "--pattern", PATTERN], check=True)


## 3. Inspect quality classes and processing errors

The main CSV retains rejected and coherent candidates. Low support is a separate quality class, not a confirmed earthquake label. Check file errors before interpreting missing detections; retain block/pick tables and the run configuration for audit.


In [ ]:
from collections import Counter
with (OUTPUT_DIR / "DAS_event_catalog.csv").open(newline="") as handle:
    events = list(csv.DictReader(handle))
print("Candidate classifications:", Counter(row["classification"] for row in events))
with (OUTPUT_DIR / "DAS_file_summary.csv").open(newline="") as handle:
    failures = [row for row in csv.DictReader(handle) if row.get("processing_error")]
print("Files with processing errors:", len(failures))
for row in failures[:10]:
    print(row.get("source_file"), row["processing_error"])


## 4. Validate separately

Compare candidates with the archived catalog and station associations using an explicitly documented timing strategy, then inspect waveforms. Record manual overrides separately. This notebook alone does not reproduce the final 44-candidate manuscript catalog, which includes subsequent decisions.
